In [1]:
# Install if required
# !pip install python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv

# Path to your .env file
env_path = Path(r"F:\AI_ATS_SCANNER\.env")

# Load environment variables
load_dotenv(dotenv_path=env_path, override=True)

# Retrieve and validate the API key
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        f"GOOGLE_API_KEY not found in {env_path}. "
        "Please check your .env file."
    )

# Set the key explicitly for Google GenAI integrations
os.environ["GOOGLE_API_KEY"] = api_key

print("GOOGLE_API_KEY loaded successfully.")

GOOGLE_API_KEY loaded successfully.


In [2]:
from pathlib import Path
from docling.document_converter import DocumentConverter
import pandas as pd

resume_folder = r"F:\AI_ATS_SCANNER\data\resumes"
output_folder = r"F:\AI_ATS_SCANNER\data\resumes\extracted_resumes"

Path(output_folder).mkdir(parents=True, exist_ok=True)

converter = DocumentConverter()

results = []

supported_extensions = {
    ".pdf", ".docx", ".pptx", ".html"
}

for file_path in Path(resume_folder).rglob("*"):

    if not file_path.is_file():
        continue

    if file_path.suffix.lower() not in supported_extensions:
        continue

    try:
        result = converter.convert(str(file_path))

        # Export document while preserving table structure
        markdown_text = result.document.export_to_markdown()

        # Save each resume's extracted content
        output_path = (
            Path(output_folder) /
            f"{file_path.stem}.md"
        )

        output_path.write_text(
            markdown_text,
            encoding="utf-8"
        )

        results.append({
            "file_name": file_path.name,
            "status": "Success",
            "character_count": len(markdown_text),
            "extracted_text": markdown_text
        })

        print(f"Processed: {file_path.name}")

    except Exception as e:
        results.append({
            "file_name": file_path.name,
            "status": f"Failed: {str(e)}",
            "character_count": 0,
            "extracted_text": ""
        })

df = pd.DataFrame(results)

print(f"\nTotal files processed: {len(df)}")
display(df[["file_name", "status", "character_count"]])

f:\AI_ATS_SCANNER\ai_ats_scanner\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 5516.99it/s]


Processed: Aditi_Sarang_Resume.pdf
Processed: Aniket_Vartak_Resume.pdf
Processed: Harshit_Jhalani_Resume.pdf
Processed: Kavya_Raghunathan_Resume.pdf
Processed: Sanika_Deshmukh_Resume.pdf
Processed: Siddharth_Bhandari_Resume.pdf

Total files processed: 6


,file_name,status,character_count
0,Aditi_Sarang_Resume.pdf,Success,2504
1,Aniket_Vartak_Resume.pdf,Success,2853
2,Harshit_Jhalani_Resume.pdf,Success,3379
3,Kavya_Raghunathan_Resume.pdf,Success,3548
4,Sanika_Deshmukh_Resume.pdf,Success,3506
5,Siddharth_Bhandari_Resume.pdf,Success,3676


In [3]:

from pathlib import Path
from datetime import date
import pandas as pd
import json
import time
import logging

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Folder containing the extracted Markdown resumes
resume_folder = Path(
    r"F:\AI_ATS_SCANNER\data\resumes\extracted_resumes"
)

# Output folder for CSV and logs
output_folder = Path(
    r"F:\AI_ATS_SCANNER\data\resumes\analysis_results"
)
output_folder.mkdir(parents=True, exist_ok=True)

# Main output CSV: one row per successfully processed resume
output_csv = output_folder / "resume_analysis.csv"

# Failed resumes are logged here for troubleshooting
error_log = output_folder / "resume_errors.log"

# Optional: pause between requests to reduce request bursts
REQUEST_DELAY_SECONDS = 1

# Validate input directory
if not resume_folder.exists():
    raise FileNotFoundError(
        f"Resume folder does not exist: {resume_folder}"
    )


# ============================================================
# 2. LOGGING CONFIGURATION
# ============================================================

logging.basicConfig(
    filename=str(error_log),
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8"
)


# ============================================================
# 3. INITIALIZE GEMINI AND JSON PARSER
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

parser = JsonOutputParser()


# ============================================================
# 4. DEFINE THE EXTRACTION PROMPT
# ============================================================


prompt = PromptTemplate(
    template="""
You are an expert resume parsing and professional experience calculation
system. Extract accurate, structured information from the resume.

RESUME:
{resume_text}

CURRENT DATE:
{current_date}

INSTRUCTIONS:

============================================================
1. PROFESSIONAL WORK EXPERIENCE
============================================================

Calculate total professional work experience using the following
strict, sequential procedure.

IMPORTANT:
- Do not guess or invent dates.
- Do not double-count overlapping employment periods.
- Use the CURRENT DATE provided above for all calculations.
- Follow the steps in order.

STEP 1: IDENTIFY EXPLICITLY MENTIONED TOTAL EXPERIENCE

First, search the entire resume for statements that explicitly mention
total professional experience.

Examples:
- "3 years of experience"
- "2 years and 6 months of experience"
- "Total experience: 4.5 years"
- "Professional experience: 36 months"

Record the explicitly mentioned total experience internally.
Do not assume that this value is correct without checking the
employment date ranges in Step 2.

If the resume does not explicitly mention total experience,
record that no explicit total was found.

Do not confuse:
- Years of experience required for a job
- Experience mentioned in a job description
- Academic experience
- Internship duration
- Project duration
with the candidate's actual total professional experience.

STEP 2: IDENTIFY ALL PROFESSIONAL EMPLOYMENT DATE RANGES

Read the resume carefully and identify every employment period
listed under sections such as:
- Professional Experience
- Work Experience
- Employment History
- Experience
- Career History

For each employment entry, extract:
- Start month and year, if available
- End month and year, if available
- Whether the role is marked Present, Current, or Ongoing

Examples:
- January 2022 - March 2023
- Jun 2023 - Dec 2024
- February 2025 - Present

Include professional full-time and part-time employment if listed.
Include freelance work only when explicitly presented as professional
employment.

Do not count education, academic projects, or internships as
professional work experience unless they are explicitly identified
as professional employment.

Do not use a stated total-experience figure as a substitute for
identifying employment date ranges.

DATE HANDLING RULES:

A. If the end date is "Present", "Current", or "Ongoing", use the
   CURRENT DATE provided above as the end date.

B. If both start and end month/year are available, use those dates.

C. If only a year is available, do not invent a month. Use January
   as the start month only when the start year alone is given, and
   December as the end month only when the end year alone is given.
   Treat these as explicit estimation conventions and apply them
   consistently.

D. If the start date is missing or unusable, do not invent it.
   Exclude that period from the date-based calculation.

E. If an employment entry contains conflicting or impossible dates,
   do not invent a correction. Exclude the invalid period.

STEP 3: CALCULATE TOTAL EXPERIENCE PRECISELY

Calculate the duration of every valid employment date range.

Use calendar months as the calculation unit.

For each date range:
- Convert the start and end dates to year-month values.
- Calculate the number of months between them:
  (end_year - start_year) * 12 + (end_month - start_month).
- If the end date is Present, use the current year and month from
  the CURRENT DATE provided above.
- Do not add an extra month merely because both endpoint months
  appear in the resume.

Example:
January 2022 - January 2023 = 12 months.
June 2023 - December 2024 = 18 months.

Then:
- Combine all valid employment periods.
- Identify overlapping periods.
- Count overlapping calendar months only once.
- Sum the non-overlapping employment months to obtain total
  professional experience in months.

If the candidate changed jobs without a gap, count the continuous
employment period normally. If there is a gap between jobs, do not
count the gap.

STEP 4: CROSS-CHECK THE EXPLICIT TOTAL

Compare the calculated date-based total from Step 3 with the
explicitly stated total from Step 1, if available.

The final result MUST be based on the date ranges identified and
calculated in Steps 2 and 3, not on the stated total.

If the stated total differs from the date-based calculation,
use the date-based calculation.

Do not add the explicitly stated total to the date-based total.
Do not count the same experience twice.

STEP 5: CONVERT TOTAL MONTHS TO YEARS AND REMAINING MONTHS

After calculating the final total number of months:

experience_years = total_months // 12
experience_months = total_months % 12

Examples:
- 6 total months = 0 years, 6 months
- 12 total months = 1 year, 0 months
- 18 total months = 1 year, 6 months
- 29 total months = 2 years, 5 months
- 36 total months = 3 years, 0 months

If no valid professional employment date ranges can be identified,
return:
- experience_years = 0
- experience_months = 0

Do not use an explicitly stated total as the final result when
there are no usable employment date ranges.

============================================================
2. SKILLS
============================================================

Return the following binary indicators:

- python: 1 if Python is mentioned anywhere in the resume,
  otherwise 0.

- sql: 1 if SQL is mentioned anywhere in the resume,
  otherwise 0.

- machine_learning: 1 if Machine Learning is explicitly mentioned
  or clearly demonstrated by the candidate's skills or work,
  otherwise 0.

Do not infer skills solely from unrelated job titles.

============================================================
3. OUTPUT REQUIREMENTS
============================================================

Return only valid JSON matching the specified output schema.

Required fields:
- experience_years: non-negative integer
- experience_months: integer from 0 to 11
- python: integer, either 0 or 1
- sql: integer, either 0 or 1
- machine_learning: integer, either 0 or 1

Do not include additional fields or explanations.

Before returning the JSON, internally verify:
1. All eligible employment periods were identified.
2. Present roles were calculated through the supplied current date.
3. Overlapping periods were counted only once.
4. The total months were converted correctly into years and months.
5. The stated total experience was not added again to the date-based
   calculation.

{format_instructions}
""",
    input_variables=["resume_text", "current_date"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

# Build the LangChain pipeline
chain = prompt | llm | parser


# ============================================================
# 5. DEFINE THE OUTPUT SCHEMA
# ============================================================

# These columns will be saved in the final CSV.
output_columns = [
    "file_name",
    "experience_years",
    "experience_months",
    "python",
    "sql",
    "machine_learning"
]



In [4]:
llm.invoke("Hi tell me about GENAI")

AIMessage(content=[{'type': 'text', 'text': '**Generative AI (GenAI)** is a branch of artificial intelligence that can create **new, original content**—including text, images, audio, video, code, and 3D models—in response to simple text prompts. \n\nUnlike traditional AI, which is designed to analyze data, classify information, or make predictions (like detecting spam or recommending a movie), **GenAI creates something that didn\'t exist before.**\n\nHere is a breakdown of everything you need to know about GenAI.\n\n---\n\n### 1. How Does It Work?\nGenAI systems are trained on massive amounts of existing data (books, websites, images, music, code). \n* **Learning Patterns:** The AI doesn\'t "copy and paste." Instead, it learns the underlying patterns, structures, and relationships in the data. \n* **Predicting the Next Element:** When you give it a prompt, it uses probability to predict what should come next. For text, it predicts the next word; for images, it predicts the next pixel.\

In [5]:

# ============================================================
# 6. LOAD ALREADY COMPLETED RESUMES
# ============================================================

# The CSV itself acts as the checkpoint.
# On restart, previously successful resumes are skipped.

if output_csv.exists():
    existing_df = pd.read_csv(output_csv)

    # Check that the CSV contains the required columns
    missing_columns = set(output_columns) - set(existing_df.columns)

    if missing_columns:
        raise ValueError(
            f"Existing CSV is missing columns: {missing_columns}. "
            "Please back up and repair the CSV before continuing."
        )

    # Only treat valid, successful records as completed.
    completed_files = set(
        existing_df.loc[
            existing_df["file_name"].notna(),
            "file_name"
        ].astype(str)
    )

else:
    # Create the CSV with headers before starting processing.
    pd.DataFrame(columns=output_columns).to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig"
    )
    completed_files = set()


# ============================================================
# 7. FIND PENDING MARKDOWN FILES
# ============================================================

# Sort files for predictable processing order.
all_md_files = sorted(resume_folder.rglob("*.md"))

# Exclude resumes already recorded in the output CSV.
pending_files = [
    file_path
    for file_path in all_md_files
    if file_path.name not in completed_files
]

print(f"Total Markdown files found : {len(all_md_files)}")
print(f"Already completed          : {len(completed_files)}")
print(f"Pending resumes            : {len(pending_files)}")
print(f"Output CSV                 : {output_csv}")


Total Markdown files found : 6
Already completed          : 6
Pending resumes            : 0
Output CSV                 : F:\AI_ATS_SCANNER\data\resumes\analysis_results\resume_analysis.csv


In [ ]:


# ============================================================
# 8. PROCESS RESUMES ONE BY ONE
# ============================================================

for index, file_path in enumerate(pending_files, start=1):

    print(
        f"\n[{index}/{len(pending_files)}] "
        f"Processing: {file_path.name}"
    )

    try:
        # ----------------------------------------------------
        # A. Read the extracted Markdown text
        # ----------------------------------------------------
        resume_text = file_path.read_text(encoding="utf-8").strip()
        # print(resume_text)
        if not resume_text:
            raise ValueError("Resume Markdown file is empty.")

        # ----------------------------------------------------
        # B. Invoke Gemini and parse the JSON response
        # ----------------------------------------------------
        response = chain.invoke({
            "resume_text": resume_text,
            "current_date": date.today().strftime("%Y-%m-%d")
        })

        # ----------------------------------------------------
        # C. Validate the returned JSON fields
        # ----------------------------------------------------
        required_keys = [
            "experience_years",
            "experience_months",
            "python",
            "sql",
            "machine_learning"
        ]

        if not isinstance(response, dict):
            raise ValueError("LLM response is not a JSON object.")

        missing_keys = set(required_keys) - set(response.keys())

        if missing_keys:
            raise ValueError(
                f"Missing JSON fields: {missing_keys}"
            )

        # Validate integer fields; reject booleans and strings.
        for key in required_keys:
            value = response[key]

            if type(value) is not int:
                raise ValueError(
                    f"{key} must be an integer; received {value!r}"
                )

        if (
            response["experience_years"] < 0
            or not 0 <= response["experience_months"] <= 11
        ):
            raise ValueError("Invalid experience years/months.")

        for key in ["python", "sql", "machine_learning"]:
            if response[key] not in (0, 1):
                raise ValueError(
                    f"{key} must be either 0 or 1."
                )

        # ----------------------------------------------------
        # D. Prepare one CSV row
        # ----------------------------------------------------
        row = {
            "file_name": file_path.name,
            "experience_years": response["experience_years"],
            "experience_months": response["experience_months"],
            "python": response["python"],
            "sql": response["sql"],
            "machine_learning": response["machine_learning"]
        }

        # ----------------------------------------------------
        # E. Append immediately after successful execution
        # ----------------------------------------------------
        # Append only one row at a time.
        # Never wait for the entire loop to finish.

        pd.DataFrame([row], columns=output_columns).to_csv(
            output_csv,
            mode="a",
            header=False,
            index=False,
            encoding="utf-8-sig"
        )

        # Update the in-memory checkpoint after the CSV write.
        completed_files.add(file_path.name)

        print("SUCCESS:", json.dumps(row, ensure_ascii=False))

    except Exception as e:
        # Log the error but continue with the remaining resumes.
        logging.exception(
            "Failed to process %s", file_path.name
        )

        print(f"FAILED: {file_path.name} | {e}")
        print("This resume can be retried in the next run.")

    # --------------------------------------------------------
    # F. Optional pause between API requests
    # --------------------------------------------------------
    if REQUEST_DELAY_SECONDS > 0:
        time.sleep(REQUEST_DELAY_SECONDS)


# ============================================================
# 9. DISPLAY FINAL RESULTS
# ============================================================

final_df = pd.read_csv(output_csv)

print("\n" + "=" * 60)
print("RESUME EXTRACTION SUMMARY")
print("=" * 60)

print(f"Markdown files found : {len(all_md_files)}")
print(f"Successfully saved   : {len(completed_files)}")
print(f"Still pending/failed : {len(all_md_files) - len(completed_files)}")

print(f"\nCSV saved at: {output_csv}")

display(final_df)



RESUME EXTRACTION SUMMARY
Markdown files found : 6
Successfully saved   : 6
Still pending/failed : 0

CSV saved at: F:\AI_ATS_SCANNER\data\resumes\analysis_results\resume_analysis.csv


,file_name,experience_years,experience_months,python,sql,machine_learning
0,Aditi_Sarang_Resume.md,0,0,1,1,1
1,Aniket_Vartak_Resume.md,7,3,0,1,0
2,Harshit_Jhalani_Resume.md,1,7,1,1,1
3,Kavya_Raghunathan_Resume.md,2,8,1,1,1
4,Sanika_Deshmukh_Resume.md,0,0,1,1,1
5,Siddharth_Bhandari_Resume.md,13,11,1,1,0


: 